# Fine-tuning de DNABERT-2 contra una CNN ajustada

Pregunta: si DNABERT-2 ajustado de punta a punta supera a ridge sobre sus embeddings congelados
(R² 0.192 / 0.178 por réplica, 0.257 contra el promedio), y si esa ganancia viene del preentrenamiento
o la consigue cualquier modelo profundo supervisado bien entrenado (la CNN).

Todo el código está en `finetuning.py` (ver su encabezado para el diseño). Resumen:

- Mismo split por cromosoma que el resto. Validación interna (~10% del entrenamiento, por cromosoma)
  para parada temprana y elección de hiperparámetros; el fold de test nunca se mira.
- Multitarea con tres salidas: percentil Rep 1, percentil Rep 2 y promedio de percentiles. Así una sola
  corrida da predicciones para los dos modos de `mejoras.py`.
- CNN: 12 configuraciones al azar por fold. DNABERT-2: lr 2e-5 y 5e-5 por fold. Con la mejor, 3
  semillas promediadas.
- Cada (modelo, fold, semilla) se guarda en Drive apenas termina: si Colab se desconecta, volver a
  correr las celdas retoma donde quedó.

**Antes de correr:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`. En la misma carpeta de
Drive del notebook de embeddings tienen que estar `embeddings_adn_dataset_v1.tsv`, `evaluar.py` y
`finetuning.py` (este último de `TesisDoc/R/71_exploratorio_embeddings_adn/`).

Tiempo estimado en T4: CNN ~20 a 30 min; DNABERT-2 ~1.5 a 2 h.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CARPETA = '/content/drive/MyDrive/embeddings_adn'   # <- cambiar si hace falta

In [ ]:
# Misma versión fijada que en el notebook de embeddings (el código propio de DNABERT-2 no anda con
# transformers nuevos) y sin triton (su atención en triton no compila; cae a PyTorch estándar).
!pip -q install einops "transformers==4.46.3"
!pip -q uninstall -y triton

**Después de la celda anterior: `Entorno de ejecución → Reiniciar sesión`**, y seguir desde la celda
siguiente (volver a correr la de Drive). Comprobar que `transformers.__version__` diga 4.46.3.

In [ ]:
import sys, torch, transformers, pandas as pd
sys.path.insert(0, CARPETA)
import evaluar as ev, finetuning as ft
df = ev.cargar(f'{CARPETA}/embeddings_adn_dataset_v1.tsv')
print(len(df), 'promotores |', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU',
      '| transformers', transformers.__version__)

## CNN ajustada

Bloques conv-BN-ReLU-maxpool-dropout y capa densa sobre el mapa aplanado (conserva posición), aumento
por desplazamiento de hasta ±3 pb. Lo que se busca por fold: filtros, kernel de la primera capa, número
de bloques, dropout, ancho de la capa densa, lr, weight decay y desplazamiento.

In [ ]:
ft.correr_cnn(df, CARPETA)

## DNABERT-2 con fine-tuning

Modelo completo ajustado (AdamW, warmup lineal 10%, hasta 8 épocas, parada temprana con paciencia 2,
lotes de 32, fp16) y cabezal lineal sobre el pooling por las mismas tres ventanas de los embeddings
congelados.

In [ ]:
ft.correr_dnabert2(df, CARPETA)

## Resumen

R² de cada modelo contra los tres objetivos. Referencias (ridge, `mejoras.py`): DNABERT-2 congelado
ventanas c3 0.192 / 0.178 por réplica y 0.257 contra el promedio; k-meros 1 a 4 por ventana 0.146 /
0.135 y 0.195. `_s0` es una sola semilla; sin sufijo, el promedio de 3.

In [ ]:
res = ft.ensamblar(df, CARPETA)
res.round(3)

## Llevar los resultados a TesisDoc

Bajar de Drive `ft/predicciones_replica/*.tsv` a
`TesisDoc/figures/exploratorio_embeddings_adn/mejoras/predicciones/` y `ft/predicciones_promedio/*.tsv` a
`TesisDoc/figures/exploratorio_embeddings_adn/mejoras_promedio/predicciones/`. Después, local (no
recalcula nada, solo rehace el resumen y el bootstrap con los modelos nuevos incluidos):

```
python R/71_exploratorio_embeddings_adn/mejoras.py
python R/71_exploratorio_embeddings_adn/mejoras.py --objetivo promedio
```

Conviene bajar también `ft/cnn_ajustada/` y `ft/dnabert2_ft/` (los `busqueda_fold*.json` tienen los
hiperparámetros elegidos y las épocas) a `figures/exploratorio_embeddings_adn/finetuning/`.